# Predicting Diabetes Prevalence Across U.S. Counties


# 1. Project Overview

This project explores the relationship between social vulnerability,
air pollution, obesity, and diabetes prevalence across U.S. counties.

Public health, social vulnerability, and environmental data were combined
at the county level to examine these relationships.

The project uses exploratory data analysis, statistical modeling,
and machine learning to understand the factors associated with diabetes
prevalence and evaluate their predictive value.

# 2. Research Question

How are social vulnerability and PM2.5 exposure associated with diabetes
prevalence across U.S. counties, and how well can these factors predict
county-level diabetes prevalence?

During model evaluation, obesity prevalence was identified as an additional
predictor and was included in later models to improve predictive performance.

# 3. Objectives

- Examine relationships between social vulnerability, PM2.5, and health outcomes.
- Evaluate the association of SVI and PM2.5 with diabetes prevalence.
- Build models to predict county-level diabetes prevalence.
- Analyze model errors to identify additional predictive factors.
- Compare machine learning models using cross-validation.
- Evaluate the final model on unseen test data.

# 4. Import Libraries

In [396]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)
from statsmodels.stats.diagnostic import het_breuschpagan

# 5. Data Sources

This project combines three public datasets at the U.S. county level:

**CDC PLACES**

*Year: 2021 - 2022*

County level health indicators, including:
- Diabetes prevalence
- Obesity prevalence
- COPD prevalence
- Asthma prevalence

Source: https://data.cdc.gov/d/d3i6-k6z5

**CDC/ATSDR Social Vulnerability Index (SVI)**

*Year: 2022*

County level measures of social vulnerability, including the overall
Social Vulnerability Index (RPL_THEMES).

Source: https://www.atsdr.cdc.gov/place-health/php/svi/svi-data-documentation-download.html?utm_source=chatgpt.com

**Daily County-Level PM2.5 Concentrations**

*Year: 2001 -2022*

Daily county-level PM2.5 estimates were aggregated to calculate annual
mean PM2.5 exposure for each county.

Source: https://data.cdc.gov/d/53mz-4zqd

County FIPS codes were used to merge the three datasets.

# 6. Data Loading and Preparation

The analysis combines health, social vulnerability, and environmental data
at the U.S. county level.

Before combining the datasets, the health and social vulnerability data were
cleaned and standardized separately.

County FIPS codes were used as the geographic identifier because they provide
a consistent way to match counties across different data sources.

In [397]:
# Load CDC PLACES data
places = pd.read_csv("PLACES__County_Data_2024.csv")
# Load SVI data
svi = pd.read_csv("SVI_2022_US_COUNTY.csv")

print("PLACES shape:", places.shape)
print("SVI shape:", svi.shape)

PLACES shape: (3144, 167)
SVI shape: (3144, 158)


## 6.1 Health Data

The health dataset contains county-level health estimates from CDC PLACES.

The analysis focused on four health outcomes:

- Asthma prevalence
- COPD prevalence
- Diabetes prevalence
- Obesity prevalence

County name, state, population, and county FIPS code were also retained for
identification and analysis.

In [398]:
# PLACES columns
print(places.columns.tolist())

['StateAbbr', 'StateDesc', 'CountyName', 'CountyFIPS', 'TotalPopulation', 'TotalPop18plus', 'ACCESS2_CrudePrev', 'ACCESS2_Crude95CI', 'ACCESS2_AdjPrev', 'ACCESS2_Adj95CI', 'ARTHRITIS_CrudePrev', 'ARTHRITIS_Crude95CI', 'ARTHRITIS_AdjPrev', 'ARTHRITIS_Adj95CI', 'BINGE_CrudePrev', 'BINGE_Crude95CI', 'BINGE_AdjPrev', 'BINGE_Adj95CI', 'BPHIGH_CrudePrev', 'BPHIGH_Crude95CI', 'BPHIGH_AdjPrev', 'BPHIGH_Adj95CI', 'BPMED_CrudePrev', 'BPMED_Crude95CI', 'BPMED_AdjPrev', 'BPMED_Adj95CI', 'CANCER_CrudePrev', 'CANCER_Crude95CI', 'CANCER_AdjPrev', 'CANCER_Adj95CI', 'CASTHMA_CrudePrev', 'CASTHMA_Crude95CI', 'CASTHMA_AdjPrev', 'CASTHMA_Adj95CI', 'CHD_CrudePrev', 'CHD_Crude95CI', 'CHD_AdjPrev', 'CHD_Adj95CI', 'CHECKUP_CrudePrev', 'CHECKUP_Crude95CI', 'CHECKUP_AdjPrev', 'CHECKUP_Adj95CI', 'CHOLSCREEN_CrudePrev', 'CHOLSCREEN_Crude95CI', 'CHOLSCREEN_AdjPrev', 'CHOLSCREEN_Adj95CI', 'COLON_SCREEN_CrudePrev', 'COLON_SCREEN_Crude95CI', 'COLON_SCREEN_AdjPrev', 'COLON_SCREEN_Adj95CI', 'COPD_CrudePrev', 'COPD_Crud

In [399]:
places_selected = places [
    [
        "StateAbbr",
        "StateDesc",
        "CountyName",
        "CountyFIPS",
        "TotalPopulation",
        "CASTHMA_AdjPrev",
        "COPD_AdjPrev",
        "DIABETES_AdjPrev",
        "OBESITY_AdjPrev",
    ]
].copy()

Only variables relevant to the project were retained. Age-adjusted prevalence
estimates were used for the health outcomes to improve comparisons across
counties with different age distributions.

## 6.2 Social Vulnerability Data

The Social Vulnerability Index (SVI) measures the relative vulnerability of
communities using social and demographic characteristics.

The overall percentile ranking, RPL_THEMES, was used as the main SVI measure
in this project.

RPL_THEMES ranges approximately from 0 to 1:

- Values closer to 0 indicate lower relative social vulnerability.
- Values closer to 1 indicate higher relative social vulnerability.

The four SVI theme rankings were also retained during exploratory analysis.

In [400]:
# SVI columns
print(svi.columns.tolist())

['ST', 'STATE', 'ST_ABBR', 'STCNTY', 'COUNTY', 'FIPS', 'LOCATION', 'AREA_SQMI', 'E_TOTPOP', 'M_TOTPOP', 'E_HU', 'M_HU', 'E_HH', 'M_HH', 'E_POV150', 'M_POV150', 'E_UNEMP', 'M_UNEMP', 'E_HBURD', 'M_HBURD', 'E_NOHSDP', 'M_NOHSDP', 'E_UNINSUR', 'M_UNINSUR', 'E_AGE65', 'M_AGE65', 'E_AGE17', 'M_AGE17', 'E_DISABL', 'M_DISABL', 'E_SNGPNT', 'M_SNGPNT', 'E_LIMENG', 'M_LIMENG', 'E_MINRTY', 'M_MINRTY', 'E_MUNIT', 'M_MUNIT', 'E_MOBILE', 'M_MOBILE', 'E_CROWD', 'M_CROWD', 'E_NOVEH', 'M_NOVEH', 'E_GROUPQ', 'M_GROUPQ', 'EP_POV150', 'MP_POV150', 'EP_UNEMP', 'MP_UNEMP', 'EP_HBURD', 'MP_HBURD', 'EP_NOHSDP', 'MP_NOHSDP', 'EP_UNINSUR', 'MP_UNINSUR', 'EP_AGE65', 'MP_AGE65', 'EP_AGE17', 'MP_AGE17', 'EP_DISABL', 'MP_DISABL', 'EP_SNGPNT', 'MP_SNGPNT', 'EP_LIMENG', 'MP_LIMENG', 'EP_MINRTY', 'MP_MINRTY', 'EP_MUNIT', 'MP_MUNIT', 'EP_MOBILE', 'MP_MOBILE', 'EP_CROWD', 'MP_CROWD', 'EP_NOVEH', 'MP_NOVEH', 'EP_GROUPQ', 'MP_GROUPQ', 'EPL_POV150', 'EPL_UNEMP', 'EPL_HBURD', 'EPL_NOHSDP', 'EPL_UNINSUR', 'SPL_THEME1', 'RPL_

In [401]:
svi_selected = svi [
    [
        "FIPS",
        "ST_ABBR",
        "COUNTY",
        "RPL_THEME1",
        "RPL_THEME2",
        "RPL_THEME3",
        "RPL_THEME4",
        "RPL_THEMES"
    ]
].copy()

## 6.3 Standardize County FIPS Codes
County FIPS codes were used to connect the health and SVI datasets.

Because FIPS codes can be stored differently across datasets, they were
converted to five character strings before merging.

Leading zeros were preserved because they are part of the county identifier.

In [402]:
places_selected["CountyFIPS"]= (
    places_selected["CountyFIPS"]
    .astype(str)
    .str.zfill(5)
)
places_selected["CountyFIPS"].head(10)

,CountyFIPS
0,17065
1,36099
2,40061
3,17097
4,20005
5,02020
6,40133
7,54055
8,27013
9,31019


In [403]:
svi_selected["FIPS"]=(
    svi_selected["FIPS"]
    .astype(str)
    .str.zfill(5)
)
svi_selected["FIPS"].head(10)

,FIPS
0,01001
1,01003
2,01005
3,01007
4,01009
5,01011
6,01013
7,01015
8,01017
9,01019


## 6.4 Combining Health and Social Vulnerability Data

The cleaned health and SVI datasets were merged using county FIPS codes.

An inner join was used so that only counties present in both datasets were
included.

In [404]:
places_fips = set(places_selected["CountyFIPS"])
svi_fips = set(svi_selected["FIPS"])

print("PLACES counties:", len(places_fips))
print("SVI counties:", len(svi_fips))
print("Matching counties:", len(places_fips & svi_fips))

PLACES counties: 3144
SVI counties: 3144
Matching counties: 3144


In [405]:
health_svi = places_selected.merge(
    svi_selected,
    left_on="CountyFIPS",
    right_on="FIPS",
    how="inner"
)
print(health_svi.shape)
display(health_svi.head())

(3144, 17)


,StateAbbr,StateDesc,CountyName,CountyFIPS,TotalPopulation,CASTHMA_AdjPrev,COPD_AdjPrev,DIABETES_AdjPrev,OBESITY_AdjPrev,FIPS,ST_ABBR,COUNTY,RPL_THEME1,RPL_THEME2,RPL_THEME3,RPL_THEME4,RPL_THEMES
0,IL,Illinois,Hamilton,17065,"7,984",10.7,7.5,10.3,37.8,17065,IL,Hamilton County,0.4213,0.6507,0.0474,0.1794,0.3029
1,NY,New York,Seneca,36099,"32,882",11.0,6.7,8.9,37.4,36099,NY,Seneca County,0.6564,0.2902,0.3535,0.7070,0.5737
2,OK,Oklahoma,Haskell,40061,"11,641",13.5,10.0,13.5,43.7,40061,OK,Haskell County,0.9160,0.9166,0.6860,0.3866,0.8387
3,IL,Illinois,Lake,17097,"709,150",9.1,5.3,10.6,31.9,17097,IL,Lake County,0.3086,0.3847,0.7941,0.6118,0.4750
4,KS,Kansas,Atchison,20005,"16,108",11.2,7.6,10.6,40.5,20005,KS,Atchison County,0.3859,0.3535,0.3824,0.7181,0.4728


In [406]:
health_svi = health_svi.drop(
    columns=["FIPS", "ST_ABBR", "COUNTY"]
)

health_svi.columns.tolist()

['StateAbbr',
 'StateDesc',
 'CountyName',
 'CountyFIPS',
 'TotalPopulation',
 'CASTHMA_AdjPrev',
 'COPD_AdjPrev',
 'DIABETES_AdjPrev',
 'OBESITY_AdjPrev',
 'RPL_THEME1',
 'RPL_THEME2',
 'RPL_THEME3',
 'RPL_THEME4',
 'RPL_THEMES']

In [407]:
print(
    "Duplicate county FIPS:",
    health_svi["CountyFIPS"].duplicated().sum()
)

Duplicate county FIPS: 0


The health and SVI datasets contained matching information for 3,144 counties,
indicating strong geographic coverage between the two sources.

# 7. PM2.5 Data Preparation

Daily county-level PM2.5 estimates for 2022 were retrieved from the CDC
Environmental Health dataset using its public API.

Only the variables needed for this project were requested:

- Year
- Date
- State FIPS
- County FIPS
- Daily mean predicted PM2.5 concentration

The daily observations were later aggregated to create one annual mean PM2.5
value for each county.

## 7.1 Load PM2.5 Data

PM2.5 data were retrieved directly from the CDC public API. Because the
dataset contains more than one million daily county level observations,
the API was queried in batches.

In [410]:
import pandas as pd
import requests

base_url = "https://data.cdc.gov/resource/53mz-4zqd.json"

all_rows = []
offset = 0
limit = 50000

while True:
    params = {
        "$select": "year,date,statefips,countyfips,pm25_mean_pred",
        "$where": "year = '2022'",
        "$limit": limit,
        "$offset": offset,
    }

    response = requests.get(base_url, params=params)
    response.raise_for_status()

    rows = response.json()

    if not rows:
        break

    all_rows.extend(rows)
    offset += limit

pm25 = pd.DataFrame(all_rows)

print(f"Download complete: {len(pm25):,} rows")

Download complete: 1,116,535 rows


## 7.2 Validate the API data

In [ ]:
print("Dataset shape:", pm25.shape)
print("\nYear:")
print(pm25["year"].value_counts())

print("\nUnique dates:", pm25["date"].nunique())

The API returned 1,116,535 daily county-level observations for 2022.

The dataset contained 365 unique dates, confirming that the complete calendar
year was included.

## 7.3 Convert data types

In [ ]:
# Convert columns to numeric
pm25 ["statefips"] = pd.to_numeric(pm25["statefips"], errors="coerce")
pm25 ["countyfips"] = pd.to_numeric(pm25["countyfips"],errors="coerce")
pm25 ["pm25_mean_pred"] = pd.to_numeric(pm25["pm25_mean_pred"],errors="coerce")

# Remove missing values
pm25_clean = pm25.dropna(
    subset=["statefips", "countyfips", "pm25_mean_pred"]
).copy()

#Convert date
pm25["date"] = pd.to_datetime(
    pm25["date"],
    format="%d%b%Y",
    errors="coerce"
)


In [ ]:
pm25_clean = pm25.dropna(
    subset=[
        "statefips",
        "countyfips",
        "pm25_mean_pred",
        "date"
    ]
).copy()

In [ ]:
print("Rows before cleaning:", len(pm25))
print("Rows after cleaning:", len(pm25_clean))

## 7.4 Create the 5-digit County FIPS

In [ ]:
# Create 5digit FIPS
pm25_clean["FIPS"] = (
    pm25_clean["statefips"].astype(int).astype(str).str.zfill(2)
    + pm25_clean["countyfips"].astype(int).astype(str).str.zfill(3)
)

In [ ]:
display(
    pm25_clean[
        ["statefips", "countyfips", "FIPS"]
    ].head()
)

State and county FIPS components were combined to create the five-digit county
FIPS identifier used by the PLACES and SVI datasets.

## 7.5 Check the full date range

In [ ]:
print("First date:", pm25_clean["date"].min())
print("Last date:", pm25_clean["date"].max())
print("Unique dates:", pm25_clean["date"].nunique())

## 7.6 Aggregate daily data to county level

In [ ]:
pm25_county = (
    pm25_clean
    .groupby("FIPS", as_index=False)
    .agg(
        PM25_Annual_Mean=("pm25_mean_pred", "mean"),
        Days_Available=("date", "nunique")
    )
)

In [ ]:
print("Counties:", pm25_county["FIPS"].nunique())
display(pm25_county.head())

pm25_county["Days_Available"].describe()

Daily PM2.5 observations were aggregated by county to calculate annual mean
PM2.5 concentration.

All 3,059 counties in the resulting PM2.5 dataset contained observations for
all 365 days of 2022.

## 7.7 Check geographic coverage

In [ ]:
health_fips = set(health_svi["CountyFIPS"])
pm25_fips = set(pm25_county["FIPS"])

print("Health/SVI counties:", len(health_fips))
print("PM2.5 counties:", len(pm25_fips))
print("Matching counties:", len(health_fips & pm25_fips))
print("Missing PM2.5 counties:", len(health_fips - pm25_fips))

The PM2.5 dataset did not have complete geographic overlap with the health and
SVI datasets.

Of the 3,144 counties available in the combined health/SVI data, 3,051 had a
matching PM2.5 record.

In [ ]:
missing_pm25 = health_fips - pm25_fips
extra_pm25 = pm25_fips - health_fips

print("Health/SVI without PM2.5:", len(missing_pm25))
print("\nPM2.5 without Health/SVI:", len(extra_pm25))


# 8. Dataset Integration

After preparing the health, social vulnerability, and PM2.5 datasets,
the data were combined at the county level using five digit County FIPS codes.

The health and SVI data contained 3,144 counties, while the annual PM2.5
dataset contained 3,059 counties.

An inner join was used so that the final analytical dataset included only
counties with matching information across all three data sources.

## 8.1 Merge Health, SVI with PM2.5

In [ ]:
final_data = health_svi.merge(
    pm25_county[["FIPS","PM25_Annual_Mean"]],
    left_on="CountyFIPS",
    right_on="FIPS",
    how="inner"
)

print("Final dataset shape:", final_data.shape)

## 8.2 Remove the Extra FIPS Column

In [ ]:
final_data = final_data.drop(columns=["FIPS"])

In [ ]:
final_data.head()

## 8.3 Validate the Final Merge

In [ ]:
print("Health/SVI counties:", health_svi["CountyFIPS"].nunique())
print("PM2.5 counties:", pm25_county["FIPS"].nunique())
print("Final counties:", final_data["CountyFIPS"].nunique())

The final dataset contained 3,051 U.S. counties with matching health,
social vulnerability, and PM2.5 information.

The reduction from 3,144 Health/SVI counties to 3,051 counties occurred
because PM2.5 data were not available for every county represented in the
health and SVI datasets.

## 8.4 Check for Duplicate Counties

In [ ]:
duplicate_counties = final_data["CountyFIPS"].duplicated().sum()

print("Duplicate counties:", duplicate_counties)

No duplicate County FIPS codes were found, confirming that each county is
represented once in the final dataset.

## 8.5 Check Missing Values

In [ ]:
analysis_columns = [
    "CASTHMA_AdjPrev",
    "COPD_AdjPrev",
    "DIABETES_AdjPrev",
    "OBESITY_AdjPrev",
    "RPL_THEMES",
    "PM25_Annual_Mean"
]

final_data[analysis_columns].isna().sum()

No missing values were present in the variables selected for the main analysis.

## 8.6 Final Analysis Variables

| Variable | Description |
|---|---|
| CASTHMA_AdjPrev | Age adjusted current asthma prevalence (%) |
| COPD_AdjPrev | Age adjusted COPD prevalence (%) |
| DIABETES_AdjPrev | Age adjusted diabetes prevalence (%) |
| OBESITY_AdjPrev | Age adjusted obesity prevalence (%) |
| RPL_THEMES | Overall Social Vulnerability Index (SVI) percentile ranking |
| PM25_Annual_Mean | Annual mean PM2.5 concentration |

Diabetes prevalence was selected as the primary outcome for statistical and
predictive modeling.

Asthma and COPD were retained for exploratory analysis to examine broader
health patterns across counties.

## 8.7 Preview the Final Dataset

In [ ]:
display(
    final_data[
        [
            "StateAbbr",
            "CountyName",
            "TotalPopulation",
            "DIABETES_AdjPrev",
            "OBESITY_AdjPrev",
            "RPL_THEMES",
            "PM25_Annual_Mean"
        ]
    ].head()
)

In [ ]:
print("Final number of counties:", len(final_data))

# 9. Exploratory Data Analysis

Exploratory Data Analysis (EDA) was conducted to understand the distribution
of health, social vulnerability, and environmental variables across the
3,051 counties in the final dataset.

The analysis focused on:

- Asthma prevalence
- COPD prevalence
- Diabetes prevalence
- Obesity prevalence
- Social Vulnerability Index (SVI)
- Annual mean PM2.5 concentration

Distributions, correlations, and pairwise relationships were examined before
statistical and predictive modeling.

In [ ]:
# Descriptive Statistics
final_data[analysis_columns].describe().round(2)

Across the counties included in the analysis, average diabetes prevalence
was approximately 11.15%, while average obesity prevalence was 38.0%.

The wide ranges observed for diabetes, obesity, and COPD indicate meaningful
variation in health outcomes across U.S. counties.

SVI values covered nearly the full 0 to 1 range, representing counties with
both low and high levels of relative social vulnerability.

## 9.2 Variable Distributions

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

labels = {
    "CASTHMA_AdjPrev": "Asthma Prevalence (%)",
    "COPD_AdjPrev": "COPD Prevalence (%)",
    "DIABETES_AdjPrev": "Diabetes Prevalence (%)",
    "OBESITY_AdjPrev": "Obesity Prevalence (%)",
    "RPL_THEMES": "Social Vulnerability Index (Percentile Rank)",
    "PM25_Annual_Mean": "Annual Mean PM2.5 Concentration (µg/m³)"
}

for column in analysis_columns:
  plt.figure(figsize=(7,4))
  sns.histplot(
      data=final_data,
      x=column,
      bins=30,
      kde=(column!="RPL_THEMES")
    )
  plt.title(f"Distribution of {labels[column]}")
  plt.xlabel(labels[column])
  plt.ylabel("Number of Counties")
  plt.show()

The health variables showed different distribution patterns across counties.

Diabetes and COPD showed noticeable right tails, indicating that a smaller
number of counties had substantially higher prevalence than the national
county distribution.

Obesity prevalence showed considerable variation across counties, while SVI
covered nearly the full range from low to high social vulnerability.

Annual mean PM2.5 concentrations were concentrated around the middle of their
observed range, with fewer counties at very low or very high levels.

## 9.3 Correlation Matrix

In [ ]:
correlation_matrix = final_data[analysis_columns].corr()
correlation_matrix.round(2)

correlation_matrix_labeled = correlation_matrix.rename(
  index=labels,
  columns=labels,
)

plt.figure(figsize=(11,8))
sns.heatmap(
    correlation_matrix_labeled,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    center=0
)
plt.title("Correlation Matrix of Health, Social Vulnerability, and PM2.5")
plt.tight_layout()

plt.savefig(
    "correlation_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

Several notable relationships were observed across counties.

Social vulnerability showed a strong positive correlation with diabetes
prevalence, indicating that counties with higher social
vulnerability tended to have higher diabetes prevalence.

Obesity and diabetes were also strongly positively correlated.

COPD showed strong relationships with several health outcomes, suggesting
that some county-level health conditions tend to occur together.

PM2.5 showed weaker relationships with the health outcomes compared with
social vulnerability and obesity. Its correlation with diabetes was positive
but more moderate.

Correlation describes association between variables and does not establish
causation.

## 9.4 Social Vulnerability vs. Diabetes Prevalence

In [ ]:
# Social Vulnerability vs Diabetes Prevalence
plt.figure(figsize=(7,5))
sns.regplot(
    data=final_data,
    x="RPL_THEMES",
    y="DIABETES_AdjPrev",
    scatter_kws={"alpha": 0.25},
    line_kws={"color": "red"}
)

plt.title("Social Vulnerability vs Diabetes Prevalence")
plt.xlabel("Social Vulnerability Index (Percentile Rank)")
plt.ylabel("Diabetes Prevalence (%)")
plt.tight_layout()
plt.show()

## 9.5 PM2.5 vs. Diabetes Prevalence

In [ ]:
# PM2.5 vs Diabetes Prevalence

plt.figure(figsize=(7,5))
sns.regplot(
    data=final_data,
    x="PM25_Annual_Mean",
    y="DIABETES_AdjPrev",
    scatter_kws={"alpha": 0.25},
    line_kws={"color": "red"}
)

plt.title("PM2.5 Exposure vs Diabetes Prevalence")
plt.xlabel("Annual Mean PM2.5 Concentration (µg/m³)")
plt.ylabel("Diabetes Prevalence (%)")
plt.tight_layout()
plt.show()

## 9.6 Obesity vs. Diabetes Prevalence

In [ ]:
# Obesity vs. Diabetes Prevalence

plt.figure(figsize=(7, 5))

sns.regplot(
    data=final_data,
    x="OBESITY_AdjPrev",
    y="DIABETES_AdjPrev",
    scatter_kws={"alpha": 0.25},
    line_kws={"color": "red"}
)

plt.title("Obesity vs. Diabetes Prevalence")
plt.xlabel("Obesity Prevalence (%)")
plt.ylabel("Diabetes Prevalence (%)")

plt.tight_layout()
plt.show()

The scatterplots reinforce the patterns observed in the correlation matrix.

Diabetes prevalence showed a clear positive relationship with social
vulnerability and obesity prevalence.

The relationship between PM2.5 and diabetes was positive but more dispersed,
suggesting that PM2.5 alone does not explain much of the variation in diabetes
prevalence across counties.

These patterns motivated the statistical analysis of SVI and PM2.5 while
considering additional health factors during later model diagnostics.

## 9.7 Pairplot

In [ ]:
pairplot_labels = {
    "CASTHMA_AdjPrev": "Asthma (%)",
    "COPD_AdjPrev": "COPD (%)",
    "DIABETES_AdjPrev": "Diabetes (%)",
    "OBESITY_AdjPrev": "Obesity (%)",
    "RPL_THEMES": "Social Vulnerability (SVI)",
    "PM25_Annual_Mean": "PM2.5 (µg/m³)"
}

pairplot_data = final_data[analysis_columns].rename(
    columns=pairplot_labels
)

g = sns.pairplot(
    pairplot_data,
    corner=True,

    plot_kws={"alpha": 0.25, "s": 20}
)

g.fig.suptitle(
    "Pairwise Relationships Among Health, Social Vulnerability, and PM2.5",
    y=1.02
)

plt.show()

The pairplot provides a broader view of pairwise relationships among the
variables. Stronger patterns are visible among several health and social
variables, while PM2.5 relationships appear more dispersed.

# 10. Statistical Analysis

The exploratory analysis showed positive relationships between diabetes
prevalence, social vulnerability, and PM2.5.

However, correlation only examines relationships between two variables at
a time. Multiple linear regression was used to examine whether SVI and
PM2.5 were associated with diabetes prevalence when considered together.

The initial model used:

- Target: Diabetes prevalence
- Predictor 1: Social Vulnerability Index (SVI)
- Predictor 2: Annual mean PM2.5 concentration

## 10.1 Initial Multiple Linear Regression

In [ ]:
import statsmodels.api as sm

X_ols = final_data[
    ["RPL_THEMES", "PM25_Annual_Mean"]
].copy()

y_ols = final_data["DIABETES_AdjPrev"].copy()

X_ols = sm.add_constant(X_ols)

ols_model = sm.OLS(
    y_ols,
    X_ols
).fit()

print(ols_model.summary())


The initial multiple linear regression explained approximately 54.6% of
the variation in county-level diabetes prevalence (R² = 0.546).

Both predictors showed statistically significant positive associations
with diabetes prevalence (p < 0.001).

Holding PM2.5 constant, a 0.10 increase in SVI was associated with an
estimated increase of approximately 0.51 percentage points in diabetes
prevalence.

Holding SVI constant, a 1 µg/m³ increase in annual mean PM2.5 was associated
with an estimated increase of approximately 0.34 percentage points in
diabetes prevalence.

These results describe statistical associations and should not be
interpreted as causal effects.

### 10.2 Regression Diagnostics

Regression diagnostics were performed to evaluate whether the assumptions
of the linear regression model were reasonable.

Residuals represent the difference between observed and predicted diabetes
prevalence:

**Residual = Observed Value - Predicted Value**

A positive residual indicates that the model underestimated diabetes
prevalence, while a negative residual indicates that the model overestimated it.

In [ ]:
# Residual vs Fitted
fitted_values = ols_model.fittedvalues
residuals = ols_model.resid

plt.figure(figsize=(8, 5))

sns.scatterplot(
    x=fitted_values,
    y=residuals,
    alpha=0.4
)

plt.axhline(
    y=0,
    color="red",
    linestyle="--"
)

plt.title("Residuals vs. Fitted Values")
plt.xlabel("Predicted Diabetes Prevalence (%)")
plt.ylabel("Residual")

plt.tight_layout()
plt.show()

The residual plot showed increasing variability at higher predicted diabetes
values, suggesting that the variance of the errors may not be constant.

This pattern motivated a formal test for heteroscedasticity.

## 10.3 Breusch-Pagan Test

In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan

bp_test = het_breuschpagan(
    ols_model.resid,
    ols_model.model.exog
)

bp_results = {
    "LM Statistic": bp_test[0],
    "LM p-value": bp_test[1],
    "F-statistic": bp_test[2],
    "F-test p-value": bp_test[3]
}

bp_results

The Breusch-Pagan test was statistically significant (LM = 184.07,
p < 0.001), providing strong evidence of heteroscedasticity.

This indicates that the variance of the regression residuals was not constant
across predicted diabetes prevalence values. Because heteroscedasticity can
affect standard errors and statistical inference, HC3 robust standard errors
were used for the regression results.

## 10.4 HC3 Robust Model

After applying HC3 robust standard errors, both SVI and PM2.5 remained
statistically significant predictors (p < 0.001).

The estimated coefficients and R² remained unchanged, while the standard
errors were adjusted to account for heteroscedasticity.

This indicates that the main statistical findings remained consistent
after correcting the inference for unequal residual variance.

In [ ]:
ols_robust = ols_model.get_robustcov_results(
    cov_type="HC3"
)

print(ols_robust.summary())

After applying HC3 heteroscedasticity robust standard errors, both social
vulnerability and PM2.5 remained statistically significant (p < 0.001).

The regression coefficients and R² remained unchanged because HC3 adjusts
the estimated standard errors rather than changing the fitted regression
relationship.

The main statistical findings therefore remained consistent after accounting
for heteroscedasticity in the inference.

## 10.5 Q-Q Plot

In [ ]:
import statsmodels.api as sm
import matplotlib.pyplot as plt

sm.qqplot(
    ols_model.resid,
    line="45",
    fit=True
)

plt.title("Q-Q Plot of Regression Residuals")
plt.tight_layout()
plt.show()


The residuals followed the theoretical distribution reasonably well through
the center of the Q-Q plot but deviated in the tails.

The upper-tail deviation was particularly interesting because it suggested
that some counties had much higher diabetes prevalence than the model could
explain using SVI and PM2.5 alone.

These large errors were investigated further.

#11. Investigating Model Errors

The initial SVI + PM2.5 regression model explained a substantial portion of
the variation in diabetes prevalence, but diagnostic analysis revealed several
large positive residuals.

A large positive residual means that observed diabetes prevalence was much
higher than the value predicted by the model.

Instead of treating these observations only as model errors, they were
examined to determine whether they shared characteristics that were missing
from the initial model.

## 11.1 Identify the largest errors

In [ ]:
residual_analysis = final_data[
    [
        "StateAbbr",
        "CountyName",
        "DIABETES_AdjPrev",
        "OBESITY_AdjPrev",
        "RPL_THEMES",
        "PM25_Annual_Mean"
    ]
].copy()

residual_analysis["Predicted_Diabetes"] = ols_model.fittedvalues
residual_analysis["Residual"] = ols_model.resid

largest_residuals = residual_analysis.nlargest(
    10,
    "Residual"
)

display(largest_residuals)



The largest positive residuals occurred in counties where observed diabetes
prevalence was substantially higher than predicted by the initial model.

For example, Kenedy County, Texas had an observed diabetes prevalence of
21.2%, while the model predicted approximately 12.7%, resulting in a
residual of about 8.5 percentage points.

Several counties with large positive residuals also had high obesity
prevalence.

## 11.2 Investigating Obesity as a Missing Predictor

Exploratory analysis had already shown a strong positive relationship
between obesity and diabetes prevalence.

In addition, the counties with the largest positive residuals from the
initial model appeared to have high obesity prevalence.

This suggested that obesity might contain predictive information that was
not captured by SVI and PM2.5 alone.

In [ ]:
residual_obesity_corr = residual_check["Residual"].corr(
    residual_check["OBESITY_AdjPrev"]
)

print(
    "Correlation between obesity and residuals:",
    round(residual_obesity_corr, 3)
)

## 11.3 Residual Obesity Relationship

Obesity prevalence had a correlation of approximately 0.605 with the
residuals from the initial model.

This positive relationship indicates that as obesity prevalence increased,
the SVI + PM2.5 model tended to underestimate diabetes prevalence by a
larger amount.

This finding suggested that obesity could provide additional predictive
information and motivated the development of a second model that included
obesity as a predictor.

This relationship does not establish that obesity caused the model errors
or that obesity alone causes differences in county-level diabetes prevalence.

# 12. Predictive Modeling

The statistical analysis identified significant associations between social
vulnerability, PM2.5, and diabetes prevalence. Residual analysis also suggested
that obesity contained additional information not captured by the initial model.

The next step was to evaluate how well these variables could predict diabetes
prevalence in counties not used to train the model.

Two linear regression models were first compared:

- Model 1: SVI + PM2.5
- Model 2: SVI + PM2.5 + Obesity

The same training and test counties were used for both models to ensure a fair
comparison.

## 12.1 Define features and target

In [ ]:
from sklearn.model_selection import train_test_split
X = final_data[
    [
        "RPL_THEMES",
        "PM25_Annual_Mean",
        "OBESITY_AdjPrev"
    ]
].copy()

y = final_data["DIABETES_AdjPrev"].copy()

The target variable was county-level diabetes prevalence.

Three potential predictors were available for predictive modeling:

- Social Vulnerability Index (SVI)
- Annual mean PM2.5 concentration
- Obesity prevalence

Although all three variables were included in the dataset used for the split,
Model 1 used only SVI and PM2.5, while Model 2 also included obesity.

## 12.2 Train/test split

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(
    features,
    y,
    test_size=0.20,
    random_state=42
)
print("Training counties:",len(X_train))
print("Testing counties:",len(X_test))


The data were divided into an 80% training set and a 20% test set.

The training set contained 2,440 counties and was used for model development.
The test set contained 611 counties and was kept separate for evaluation.

A fixed random state was used to make the split reproducible.

## 12.3 Model 1: SVI + PM2.5

The first predictive model used the same predictors as the initial statistical
model: social vulnerability and annual mean PM2.5.

This model served as a baseline for evaluating whether adding obesity improved
prediction.

In [ ]:
from sklearn.linear_model import LinearRegression

model_1 = LinearRegression()
model_1.fit(
    X_train[["RPL_THEMES", "PM25_Annual_Mean"]],
    y_train
)

pred_1 = model_1.predict(
    X_test[["RPL_THEMES", "PM25_Annual_Mean"]]
)

## 12.4 Model 2: Adding Obesity

Residual analysis suggested that the baseline model tended to underestimate
diabetes prevalence in counties with higher obesity prevalence.

Obesity was therefore added as a third predictor to test whether it improved
predictive performance.

In [ ]:
model_2 = LinearRegression()
model_2.fit(
    X_train,
    y_train
)
pred_2 = model_2.predict(X_test)

## 12.5 Evaluate both models

In [ ]:
from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    mean_absolute_error
)
import numpy as np

model_1_results = {
    "R2": r2_score(y_test, pred_1),
    "MAE": mean_absolute_error(y_test, pred_1),
    "RMSE": np.sqrt(mean_squared_error(y_test, pred_1))
}

model_2_results = {
    "R2": r2_score(y_test, pred_2),
    "MAE": mean_absolute_error(y_test, pred_2),
    "RMSE": np.sqrt(mean_squared_error(y_test, pred_2))
}

comparison = pd.DataFrame(
    [model_1_results, model_2_results],
    index=["SVI + PM2.5", "SVI + PM2.5 + Obesity"]
)

comparison.round(3)

Adding obesity substantially improved predictive performance.

The baseline SVI + PM2.5 model achieved an R² of 0.559. After obesity was
added, R² increased to 0.757.

Prediction errors also decreased:

- MAE decreased from 1.204 to 0.885 percentage points.
- RMSE decreased from 1.524 to 1.132.

These results support the pattern identified during residual analysis:
obesity provided additional predictive information about county-level diabetes
prevalence beyond SVI and PM2.5.

## 12.6 Actual vs. Predicted

In [ ]:
# Model 1
plt.figure(figsize=(7,5))
sns.scatterplot(
    x=y_test,
    y=pred_1,
    alpha=0.5
)

min_val = min(y_test.min(), pred_1.min())
max_val = max(y_test.max(), pred_1.max())

plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    "r--"
  )
plt.title("Model 1: Actual vs. Predicted Diabetes Prevalence")
plt.xlabel("Actual Diabetes Prevalence (%)")
plt.ylabel("Predicted Diabetes Prevalence (%)")
plt.tight_layout()
plt.show()

In [ ]:
# Model 2
plt.figure(figsize=(7,5))
sns.scatterplot(
    x=y_test,
    y=pred_2,
    alpha=0.5
)

min_val = min(y_test.min(), pred_2.min())
max_val = max(y_test.max(), pred_2.max())

plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    "r--"
  )
plt.title("Model 2: Actual vs. Predicted Diabetes Prevalence")
plt.xlabel("Actual Diabetes Prevalence (%)")
plt.ylabel("Predicted Diabetes Prevalence (%)")
plt.tight_layout()
plt.savefig(
    "actual_vs_predicted.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

The Actual vs. Predicted plots show that Model 2 predictions are more closely
aligned with the perfect prediction line.

However, some of the counties with the highest diabetes prevalence continue
to be underestimated, suggesting that additional patterns may not be fully
captured by linear regression.

# 13. Cross-Validation and Model Comparison

Although adding obesity substantially improved Linear Regression, the
Actual vs. Predicted and residual plots suggested that some patterns were
still not fully captured by a linear model.

Three regression algorithms were therefore compared:

- Linear Regression
- Random Forest Regression
- Gradient Boosting Regression

All three models used the same predictors: SVI, PM2.5, and obesity prevalence.

Five-fold cross-validation was performed using only the training data to
compare model performance across multiple validation splits.

## 13.1 Why Cross-Validation?

Performance from a single data split can depend on which observations happen
to be included in the training and validation sets.

Five-fold cross-validation divides the training data into five groups. The
model is trained five times, using a different group for validation each time.

The average performance across the five folds provides a more stable estimate
of model performance during model selection.

## 13.2 Define the models

In [ ]:
# Set up the three models
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

models = {
    "Linear Regression": LinearRegression(),

    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    )
}

The models were initially compared without extensive hyperparameter tuning.
This allowed the analysis to focus on whether different modeling approaches
provided meaningful improvements over the linear baseline.

## 13.3 Set up 5-fold CV

In [ ]:
# Create the 5 folds
from sklearn.model_selection import KFold

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

## 13.4 Evaluation Metrics

Three metrics were used:

- R²: measures the proportion of variation in diabetes prevalence explained
  by the model. Higher values indicate better performance.
- MAE: measures the average absolute prediction error in percentage points.
  Lower values are better.
- RMSE: measures prediction error while giving more weight to larger errors.
  Lower values are better.

## 13.5 Run cross-validation

In [ ]:
# Run cross-validation
from sklearn.model_selection import cross_validate

scoring = {
    "R2": "r2",
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error"
}

cv_results = {}

for name, model_cv in models.items():

    scores = cross_validate(
        model_cv,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring
    )

    cv_results[name] = {
        "R2": scores["test_R2"].mean(),
        "MAE": -scores["test_MAE"].mean(),
        "RMSE": -scores["test_RMSE"].mean()
    }

In [ ]:
import pandas as pd

cv_table = pd.DataFrame(cv_results).T

cv_table = cv_table.round(3)

cv_table

Gradient Boosting showed the best average performance across the five
validation folds.

It achieved the highest mean R² (0.770) and the lowest MAE (0.842) and
RMSE (1.094).

Random Forest also performed slightly better than Linear Regression, but
Gradient Boosting provided the strongest overall results.

Based on cross-validation performance, Gradient Boosting was selected as
the final predictive model.

# 14. Final Model Evaluation

Gradient Boosting was selected as the final model based on its performance
during five-fold cross-validation.

The model was trained using the full training set and evaluated on the
held-out test set.

The test set was not used during the cross-validation model comparison.

In [ ]:
# Grading
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

final_model = GradientBoostingRegressor(
    random_state=42
)

final_model.fit(X_train, y_train)

final_pred = final_model.predict(X_test)

final_r2 = r2_score(y_test, final_pred)
final_mae = mean_absolute_error(y_test, final_pred)
final_rmse = np.sqrt(mean_squared_error(y_test, final_pred))

print(f"R2: {final_r2:.3f}")
print(f"MAE: {final_mae:.3f}")
print(f"RMSE: {final_rmse:.3f}")

The final Gradient Boosting model achieved an R² of 0.795 on the test set,
meaning that it explained approximately 79.5% of the variation in diabetes
prevalence among the test counties.

The MAE was 0.817, meaning that predictions differed from observed diabetes
prevalence by approximately 0.82 percentage points on average.

The RMSE was 1.040, indicating that some larger prediction errors remained,
but overall predictive performance was stronger than the linear baseline.

## 14.1 Compare CV vs final test

In [ ]:
performance_summary = pd.DataFrame({
    "Evaluation": [
        "5 Fold CV Average",
        "Final Test"
    ],
    "R2": [
        0.770,
        final_r2
    ],
    "MAE": [
        0.842,
        final_mae
    ],
    "RMSE": [
        1.094,
        final_rmse
    ]
})

performance_summary.round(3)

Performance on the held-out test set was similar to, and slightly better
than, the average cross-validation performance.

This suggests that the model's performance was not limited to a single
validation fold and generalized reasonably well to the held-out test counties.

## 14.2 Final Actual vs. Predicted plot

In [ ]:
plt.figure(figsize=(7, 5))

sns.scatterplot(
    x=y_test,
    y=final_pred,
    alpha=0.5
)

min_val = min(y_test.min(), final_pred.min())
max_val = max(y_test.max(), final_pred.max())

plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    "r--"
)

plt.title("Gradient Boosting: Actual vs. Predicted Diabetes Prevalence")
plt.xlabel("Actual Diabetes Prevalence (%)")
plt.ylabel("Predicted Diabetes Prevalence (%)")

plt.tight_layout()

plt.savefig(
    "actual_vs_predicted.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

The final Gradient Boosting model showed strong agreement between observed and predicted diabetes prevalence. Most predictions were close to the perfect prediction line, although the model tended to underestimate some counties with very high diabetes prevalence.

## 14.3 Final residual plot

In [ ]:
final_residuals = y_test - final_pred

plt.figure(figsize=(8, 5))

sns.scatterplot(
    x=final_pred,
    y=final_residuals,
    alpha=0.5
)

plt.axhline(
    y=0,
    color="red",
    linestyle="--"
)

plt.title("Gradient Boosting: Residuals vs. Predicted Values")
plt.xlabel("Predicted Diabetes Prevalence (%)")
plt.ylabel("Residuals")

plt.tight_layout()
plt.show()

Residuals were generally centered around zero, indicating that the final model did not show a strong overall tendency to overpredict or underpredict diabetes prevalence. Most errors were relatively small, although larger residuals remained for some counties, particularly at higher predicted prevalence levels.

# 15. Feature Importance

After evaluating the final Gradient Boosting model, feature importance was
examined to understand which predictors contributed most to the model's
predictions.

The final model included three predictors:

- Social Vulnerability Index (SVI)
- Obesity prevalence
- Annual mean PM2.5 concentration

Feature importance measures how useful each variable was to the Gradient
Boosting model when making predictions. It represents predictive contribution
and should not be interpreted as causation.

## 15.1 Calculate Feature Importance

In [ ]:
# Feature importance
import pandas as pd

feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": final_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

feature_importance

Social vulnerability was the most important feature in the final model,
with a relative importance of approximately 0.578.

Obesity prevalence was the second most important feature at approximately
0.366, while PM2.5 had a substantially smaller relative importance of
approximately 0.057.

## 15.2 Feature Importance Visualization

In [ ]:
feature_importance["Feature"] = feature_importance["Feature"].replace({
    "RPL_THEMES": "Social Vulnerability (SVI)",
    "OBESITY_AdjPrev": "Obesity Prevalence",
    "PM25_Annual_Mean": "PM2.5"
})

plt.figure(figsize=(8, 5))

sns.barplot(
    data=feature_importance,
    x="Importance",
    y="Feature"
)

plt.title("Feature Importance in Final Gradient Boosting Model")
plt.xlabel("Relative Importance")
plt.ylabel("")

plt.tight_layout()
plt.savefig(
    "feature_importance.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

The feature importance results show that social vulnerability was the strongest
predictor in the final Gradient Boosting model, followed by obesity prevalence.

PM2.5 contributed less predictive information compared with the social and
health variables.

These results suggest that differences in county-level diabetes prevalence were
more strongly predicted by social vulnerability and obesity than by PM2.5
within this model and dataset.

### **Interpretation**

The results provide an interesting perspective on the original research
question.

PM2.5 showed a positive relationship with diabetes prevalence during the
exploratory and statistical analyses. However, in the final predictive model,
its relative importance was much smaller than that of social vulnerability
and obesity.

Social vulnerability remained the most important predictor, while obesity
provided substantial additional predictive information after being identified
through residual analysis.

This suggests that social and health characteristics provided more predictive
information about county level diabetes prevalence than PM2.5 exposure among
the variables examined in this project.

These findings represent predictive relationships at the county level and
should not be interpreted as causal effects at the individual level.

# 16. Key Findings

This project examined county level differences in diabetes prevalence using
health, social vulnerability, and environmental data across the United States.

The main findings were:

- Social vulnerability showed a strong positive relationship with diabetes
  prevalence. Counties with higher SVI values generally had higher diabetes
  prevalence.

- PM2.5 was positively associated with diabetes prevalence in the initial
  statistical analysis. However, its relationship was weaker than those
  observed for social vulnerability and obesity.

- The initial regression model using SVI and PM2.5 explained approximately
  54.6% of the variation in diabetes prevalence. Both predictors remained
  statistically significant after using HC3 robust standard errors to account
  for heteroscedasticity.

- Investigation of model residuals showed that counties with higher obesity
  prevalence tended to have larger underprediction errors. The correlation
  between obesity prevalence and the initial model residuals was approximately
  0.605.

- Adding obesity substantially improved prediction. Linear Regression R²
  increased from 0.559 to 0.757, while MAE decreased from 1.204 to 0.885.

- Gradient Boosting produced the strongest average performance among the
  models evaluated using five-fold cross-validation.

- The final Gradient Boosting model achieved an R² of 0.795, MAE of 0.817,
  and RMSE of 1.040 on the test set.

- Social vulnerability had the highest feature importance in the final model,
  followed by obesity. PM2.5 had a substantially smaller relative contribution
  to prediction.

Overall, social vulnerability and obesity provided the strongest predictive
information about county-level diabetes prevalence among the variables
examined.

# 17. Limitations

Several limitations should be considered when interpreting the results of this
project.

**County-Level Analysis**

The analysis was conducted at the county level. Relationships observed between
county characteristics cannot be assumed to represent relationships between
individual people. A county with high obesity and diabetes prevalence does not
mean that the same individuals have both conditions.

**Association Does Not Establish Causation**

The statistical and machine-learning models identify associations and
predictive relationships. They do not demonstrate that social vulnerability,
obesity, or PM2.5 directly causes higher diabetes prevalence.

**Limited Predictors**

The final model included only social vulnerability, PM2.5, and obesity.
Diabetes is influenced by many additional factors that were not directly
included, such as age, physical activity, healthcare access, food environment,
income, and other demographic, behavioral, and environmental characteristics.

**SVI Is a Composite Measure**

The overall Social Vulnerability Index combines multiple social and demographic
dimensions. Although SVI was the most important predictor in the final model,
this analysis does not determine which individual components of social
vulnerability contributed most to prediction.

**Geographic Coverage**

The original Health/SVI dataset contained 3,144 counties, but only 3,051
counties had matching PM2.5 information and were included in the final analysis.
Therefore, the analytical dataset does not represent every county available in
the health data.

**Time and Data Alignment**

The datasets represent population-level estimates collected or released through
different public health data systems. Differences in measurement periods,
estimation methods, and data collection procedures may affect comparisons
between variables.

**Extreme Diabetes Values**

Although the final Gradient Boosting model performed well overall, larger
prediction errors remained for some counties, particularly counties with very
high diabetes prevalence.

**Feature Importance**

Gradient Boosting feature importance indicates how useful each variable was for
prediction within this specific model. It should not be interpreted as causal
importance or as the percentage of diabetes attributable to each predictor.

# 18. Conclusion

This project combined health, social vulnerability, and environmental data to
examine differences in diabetes prevalence across U.S. counties.

Exploratory and statistical analyses showed that social vulnerability and
PM2.5 were positively associated with diabetes prevalence. However, diagnostic
analysis of the initial SVI + PM2.5 model revealed that obesity contained
important additional information, particularly among counties where diabetes
prevalence was substantially underestimated.

Adding obesity improved predictive performance, and comparison of multiple
machine learning approaches showed that Gradient Boosting provided the strongest
overall results. The final model achieved an R² of 0.795 on the test set, with
an average absolute prediction error of approximately 0.82 percentage points.

Social vulnerability provided the greatest predictive contribution in the
final model, followed by obesity, while PM2.5 contributed less predictive
information when the three variables were considered together.

Overall, the results suggest that county level differences in diabetes
prevalence are strongly connected with social and health conditions. The
project also demonstrates how exploratory analysis, regression diagnostics,
residual investigation, and machine learning can work together to move from
an initial research question toward a stronger predictive model.